## LUAD DGE Data Process with Raw Counts

In [1]:
## import the packages

import pandas as pd 
import numpy as np
import os

/home/arraygen/.local/lib/python3.10/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/arraygen/.local/lib/python3.10/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (


In [ ]:
Project_Dir = "Cancer_Biomarkers_LUAD/ML_Implemenation_Step-2"
os.chdir(Project_Dir)

In [3]:
df = pd.read_csv("DATA/LUAD_MERGED_ALL_DEG_RESULT.csv")

In [5]:
df.head()

,ID,Control_vs_TCGA.73.4658.01A.01R.1755.07_LOG2FC,Control_vs_TCGA.73.4658.01A.01R.1755.07_P_VALUE,Control_vs_TCGA.55.8615.01A.11R.2403.07_LOG2FC,Control_vs_TCGA.55.8615.01A.11R.2403.07_P_VALUE,Control_vs_TCGA.97.8177.01A.11R.2287.07_LOG2FC,Control_vs_TCGA.97.8177.01A.11R.2287.07_P_VALUE,Control_vs_TCGA.67.3771.01A.01R.0946.07_LOG2FC,Control_vs_TCGA.67.3771.01A.01R.0946.07_P_VALUE,Control_vs_TCGA.49.6744.01A.11R.1858.07_LOG2FC,...,Control_vs_TCGA.64.1678.01A.01R.0946.07_LOG2FC,Control_vs_TCGA.64.1678.01A.01R.0946.07_P_VALUE,Control_vs_TCGA.78.7155.01A.11R.2039.07_LOG2FC,Control_vs_TCGA.78.7155.01A.11R.2039.07_P_VALUE,Control_vs_TCGA.78.7220.01A.11R.2039.07_LOG2FC,Control_vs_TCGA.78.7220.01A.11R.2039.07_P_VALUE,Control_vs_TCGA.80.5611.01A.01R.1628.07_LOG2FC,Control_vs_TCGA.80.5611.01A.01R.1628.07_P_VALUE,Control_vs_TCGA.93.8067.01A.11R.2287.07_LOG2FC,Control_vs_TCGA.93.8067.01A.11R.2287.07_P_VALUE
0,ENSG00000000003.15,1.764617,1.347665e-03,0.788529,0.152304,1.122782,0.041325,-0.163698,7.668023e-01,0.967616,...,2.434992,9.491239e-06,1.840006,8.475668e-04,1.023370,6.320607e-02,1.403937,1.077042e-02,0.617881,2.615694e-01
1,ENSG00000000005.6,6.727891,3.487233e-13,-3.546095,0.117539,1.780492,0.089232,-0.956044,5.828398e-01,2.858728,...,-2.674629,2.379395e-01,2.167960,4.774786e-02,-3.675814,1.046994e-01,-1.016395,5.593243e-01,-4.322327,5.632541e-02
2,ENSG00000000419.13,0.067779,8.600004e-01,-0.818428,0.033813,0.449834,0.240152,0.825593,3.131299e-02,0.220658,...,2.343342,7.597750e-10,0.316461,4.084877e-01,1.132402,3.078726e-03,0.736760,5.425090e-02,0.460315,2.286562e-01
3,ENSG00000000457.14,0.186545,4.915734e-01,0.146074,0.588321,0.102718,0.699836,0.446385,9.693895e-02,0.359420,...,0.657103,1.552819e-02,0.064215,8.137252e-01,0.773036,3.772405e-03,0.664009,1.329190e-02,0.607899,2.206258e-02
4,ENSG00000000460.17,0.872176,2.592718e-03,1.017284,0.000331,0.799840,0.003989,1.842173,3.062321e-11,1.083517,...,3.132582,1.424668e-30,3.230266,6.266030e-33,2.729448,7.148610e-24,2.495939,4.596216e-20,2.092399,1.151164e-14


In [6]:
PVAL_COL = [col for col in df.columns if col.endswith("_P_VALUE")]
select_col = ['ID'] + PVAL_COL
df = df[select_col]
filtered_df = df[df[PVAL_COL].le(0.05).sum(axis=1) >= len(PVAL_COL)*0.7]
print("Total Genes Selected:",filtered_df.shape[0])

Total Genes Selected: 2046


In [7]:

NORMAL_DIR = "DATA/COUNTS/NORMAL"
TUMOR_DIR = "DATA/COUNTS/TUMOR"

def MERGE_DFS(DIR_PATH):
    df_list = []
    for file in os.listdir(DIR_PATH):
        if file.endswith('.txt'):
            df = pd.read_csv(os.path.join(DIR_PATH, file), sep= "\t")
            df_list.append(df)
    merged_df = pd.concat(df_list, axis=1)
    return merged_df

NORMAL_COUNT = MERGE_DFS(NORMAL_DIR)

TUMOR_COUNT = pd.read_csv(os.path.join(TUMOR_DIR, "Tumor_LUAD_Counts.txt"), sep = "\t")




In [8]:

NORMAL_FEA = NORMAL_COUNT.T
TUMOR_FEA = TUMOR_COUNT.T
NORMAL_FEA["Label"] = 0
TUMOR_FEA["Label"] = 1

DATA_MAT = pd.concat([TUMOR_FEA, NORMAL_FEA])



In [12]:

SIG_GENES = list(filtered_df['ID'].values)
DATA_MAT_1 = DATA_MAT[SIG_GENES]
DATA_MAT_1["Label"] = DATA_MAT["Label"]
DATA_MAT_1.to_csv("DATA/LAUD_FEATURE_DATA_MATRIX.csv")


/tmp/ipykernel_3385218/2510343016.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  DATA_MAT_1["Label"] = DATA_MAT["Label"]
